In [125]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Input data is in MWh. It is divided by $10^3$ to convert it to GWh.

In [126]:
demand_base = pd.read_csv('europe_demand_2006-2015.csv', parse_dates=['datetime']).set_index('datetime')[['NO03', 'NO11', 'NO15', 'NO18', 'NO30', 'NO34', 'NO38', 'NO42', 'NO46', 'NO50', 'NO54']]/1000

# Scale demand

In [127]:
average_demand = demand_base.sum().sum()/(demand_base.index.year.to_series().unique().size)
average_demand/1000

133.8367282605634

In [128]:
demand_scaled = demand_base*(140000/average_demand) # scale the demand so that the average equals the demand of 2022 (140 TWh)

In [129]:
demand = demand_scaled.copy()

In [130]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

140.0

# Regular consumption and losses
Subtract 7 TWh (-5% of each hour in each county)

In [131]:
demand = demand - 0.05*demand

In [132]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

132.99999999999997

# Electric transport
Add 14 TWh (This is excluding electric cars which is modelled through the EV flex module in highRES. If the EV flex module is not used, demand from electric cars must be added here which should be 9 TWh).

In [ ]:
# OLD CODE
# demand = demand + np.ones(11)*((23*10**3)/(365*24*11))
# base_load = ([4.088515615, 4.880937396, 2.40344075, 1.829410414, 18.6444199, 6.106533718, 7.141278395, 4.349801459, 6.770179305, 4.947604974, 1.851576703])
# load_profile = np.array([[0.049621531], [0.033761865], [0.022347711], [0.014898474], [0.009852217], [0.006848492], [0.004805959], [0.004445512], [0.007929833], [0.008410429], [0.008410429], [0.011294005], [0.015018623], [0.018863391], [0.023068605], [0.046737955], [0.092154271], [0.102607233], [0.085906524], [0.0887901], [0.094076655], [0.091793824], [0.085786375], [0.072569987]])
# load_profile_fylke = load_profile*base_load
# for i in range(0,24):
#     demand.loc[demand.index.hour == i] = demand[demand.index.hour == i] + load_profile_fylke[i]

In [133]:
dist = pd.read_csv('2024-04-12 15-11-52 - eksport fra SINTEF energikart, fylker (2060 kollektiv og gods, strømforbruk Wt per døgn).csv', sep=';')
dist = dist.set_index('NO' + dist['county number']).drop(columns=['total number of cells', 'number of sources', 'county number', 'county name'], index=['NOUnknown'])

In [134]:
transport_demand = ((14*(10**3))/(365*24))*(dist/dist.sum())

In [135]:
demand = demand.add(transport_demand.T.values)

In [136]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

147.00972409801275

# Industry
Add 70 TWh

In [137]:
# from https://www.ssb.no/statbank/table/10314/tableViewLayout1/ (data from 2019, later years has no data for Nordland)
electricity_industry = pd.Series({'NO03':888.8, 'NO11':6424.1, 'NO15':8867.4, 'NO18':6348.9, 'NO30':4142.4, 'NO34':1091.6, 'NO38':3975.6, 'NO42':4581.8, 'NO46':12720.9, 'NO50':3946.9, 'NO54':3285.1})

In [138]:
industry = (((electricity_industry/electricity_industry.sum())*70*10**3)/(365*24)).to_numpy()

In [116]:
#industry = [0.074396516, 0.960092477, 0.819807236, 0.958079203, 0.47070081, 0.109119971, 1.142689061, 0.471555006, 1.893724859, 0.526302915, 0.564399524]

In [139]:
demand = demand + industry

In [140]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

217.0583445880764

# Petroleum
Add 6 TWh

In [141]:
electricity_petroleum = pd.Series({'NO03':0, 'NO11':3700, 'NO15':2050, 'NO18':0, 'NO30':0, 'NO34':0, 'NO38':0, 'NO42':200, 'NO46':4900, 'NO50':1600, 'NO54':4400})

In [142]:
petroleum = ((electricity_petroleum/electricity_petroleum.sum())*6000)/(365*24)

In [146]:
demand = demand + petroleum

In [147]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

223.0625120586533

# Battery production and data centres
Add 29 TWh

In [147]:
batt_n_data = [0.133872976, 0.47633873, 0.133872976, 0.282274803, 0.133872976, 0.282274803, 0.738895807, 0.430676629, 0.133872976, 0.430676629, 0.133872976]

In [148]:
demand = demand + batt_n_data

In [149]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

238.07293071796363

# To CSV

In [150]:
demand = demand*1000 # convert back to MWh

In [151]:
(demand.sum().sum())/(demand_base.index.year.to_series().unique().size) # calculate average (MWh)

238072930.71796358

In [152]:
demand.to_csv('demand_2050(no-EV)(MWh).csv')

# Directly to .dd file

For 2010

In [46]:
demand2010 = demand.loc['2010']

In [47]:
demand2010 = demand2010.reset_index().drop(columns=['datetime']).melt(ignore_index=False)

In [48]:
demand2010 = demand2010.set_index(demand2010.variable + '.' + demand2010.index.astype(str)).drop(columns=['variable']).reset_index() #.rename_axis('variable')

In [52]:
demand2010['value'] = demand2010['value']*10**3

In [53]:
demand2010

,index,value
0,NO03.0,1393.731769
1,NO03.1,1375.844302
2,NO03.2,1368.689316
3,NO03.3,1369.881813
4,NO03.4,1405.954871
...,...,...
96355,NO54.8755,1064.801134
96356,NO54.8756,1043.336174
96357,NO54.8757,1002.393751
96358,NO54.8758,964.929446


In [39]:
demand2010.to_csv('BASE_demand_2010.dd', sep=' ', lineterminator='\n')

In [54]:
    np.savetxt("BASE_demand_2010_2.dd", demand2010, delimiter=" ", fmt="%s", header="parameter \ndemand / ", footer="/ \n ", comments="")

In [ ]:
demand2010.sum()/10**6

# Read .dd file

In [ ]:
read = pd.read_csv('BASE_demand_2010.dd', sep=' ', skiprows=[0,1,96362,96363], header=None)

In [ ]:
read.mean()

In [ ]:
demand2010.mean()